# Chapter 13 — Evaluate the Agent and Log Metrics

This notebook loads the version currently marked as the **challenger**,
runs it against test data, and logs metrics to the experiment. The
promotion decision in `04_tracking_promotion` promotes the exact version
each run evaluated — it reads the `model_version` parameter logged here.

Run this notebook multiple times with different parameters (temperature,
prompt version, model version) to create candidate runs for comparison.

**Continues from:** `01_model_packaging` (model registered in Unity Catalog)


In [ ]:
%pip install mlflow
%restart_python


## 1. Configuration


In [ ]:
# The table the agent analyzes (sample input payload)
DATA_CATALOG = "demo"
DATA_SCHEMA  = "finance"
TABLE        = "sales_transactions"

# Where the agent itself is registered in Unity Catalog.
# This varies per environment; in the bundle it comes from the catalog variable.
UC_CATALOG = "dev_catalog"
UC_SCHEMA  = "finance"

try:
    username = spark.conf.get("spark.databricks.notebook.userName")
except:
    username = "unknown"
if username == "unknown":
    username = spark.sql("SELECT current_user()").collect()[0][0]
print(f"Current user: {username}")


## 2. Load the Challenger Version

We evaluate the version currently marked as the challenger, not whichever
version happens to be latest. If nothing is marked as challenger yet,
point the alias at the latest registered version first.


In [ ]:
import mlflow
import pandas as pd
import time
from mlflow import MlflowClient

client = MlflowClient()
model_name = f"{UC_CATALOG}.{UC_SCHEMA}.data_quality_agent"

# The challenger alias marks the candidate being evaluated. If it has not
# been assigned yet, point it at the latest registered version.
try:
    client.get_model_version_by_alias(model_name, "challenger")
except Exception:
    versions = client.search_model_versions(f"name='{model_name}'")
    latest = max(int(v.version) for v in versions)
    client.set_registered_model_alias(model_name, "challenger", latest)
    print(f"Assigned 'challenger' alias to version {latest}")

mlflow.set_experiment(f"/Users/{username}/data_quality_agent_experiment")

candidate = client.get_model_version_by_alias(model_name, "challenger")
model = mlflow.pyfunc.load_model(f"models:/{model_name}/{candidate.version}")
print(f"Evaluating version {candidate.version} ('challenger')")


## 3. Run the Agent and Log Metrics

We time the prediction, count the rules generated, and log everything as
an MLflow run. The `model_version` parameter records exactly which
registered version this run evaluated, so the promotion gate in
`04_tracking_promotion` can promote that same version.


In [ ]:
test_input = pd.DataFrame({
    "table_name": [TABLE], "catalog": [DATA_CATALOG], "schema": [DATA_SCHEMA]
})

with mlflow.start_run(run_name="agent_eval_v1") as run:
    mlflow.log_param("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct")
    mlflow.log_param("temperature", 0.1)
    mlflow.log_param("model_version", candidate.version)

    start = time.time()
    result = model.predict(test_input)
    elapsed = time.time() - start

    import json
    rules = json.loads(result["rules"].iloc[0])

    mlflow.log_metric("rules_generated", len(rules))
    mlflow.log_metric("latency_seconds", elapsed)

    print(f"Generated {len(rules)} rules in {elapsed:.2f}s")
    print(f"Run ID: {run.info.run_id}")


## 4. (Optional) Run Again with Different Parameters

To create multiple candidate runs for the promotion decision in
`04_tracking_promotion`, reassign the challenger alias or change
parameters (temperature, prompt version, LLM endpoint) and re-run.


## Next Steps

Multiple evaluation runs now exist in the experiment with
`rules_generated` and `latency_seconds` metrics, and each run records
the version it evaluated in `params.model_version`. Use
**04_tracking_promotion** to query these runs and promote the winner.
